In [1]:
import itertools
import numpy as np
from collections import defaultdict
from typing import Callable, Optional
import csv
import pandas as pd
import os
import utils

import utils


def is_valid_data_file(file_name:str) -> bool:
    return file_name.endswith("json") or file_name.endswith("txt")


def get_mean_for_combinations(df: pd.DataFrame, 
                       independent_variables: list[str], 
                       dependent_variables: list[str]) -> pd.DataFrame:

    # ensure all the columns are present in the df
    for col in independent_variables+dependent_variables:
        if col not in df:
            raise Exception(f"The column {col} is not in the dataframe\n\t(columns are {list(df.columns)})")
    assert(all(col in df for col in independent_variables))
    assert(dependent_variable in df for dependent_variable in dependent_variables)
    
    grouped = df.groupby(independent_variables, dropna=False)[dependent_variables].mean().reset_index()
    
    return grouped


import json
import os


def json_to_entries(data: dict):
    def item_to_list_of_entries(item) -> list[dict]:
        problem_name = item["problem_name"]
        pRef_method = item["pRef_method"]
        pRef_size = item["sample_size"]

        entries = item["results_by_tree"]

        def get_modified_entry(entry):
            entry["problem"] = problem_name
            entry["pRef_method"] = pRef_method
            entry["pRef_size"] = pRef_size

            errors = entry["results"]
            entry = entry | errors
            del entry["results"]

            if "order_tree" in entry:
                del entry["order_tree"]

            return entry

        entries = list(map(get_modified_entry, entries))
        return entries

    return [entry for item in data for entry in item_to_list_of_entries(item)]

def convert_accuracy_data_to_df(input_directory, output_filename):

    all_dicts = []
    # Iterate through all files in the input directory
    for filename in os.listdir(input_directory):
        # Construct full file path
        file_path = os.path.join(input_directory, filename)

        # Check if the file is a JSON file
        if not os.path.isfile(file_path):
            continue

        if not is_valid_data_file(file_path):
            continue

        with open(file_path, 'r') as file:
            data = json.load(file)
            entries = json_to_entries(data)
            all_dicts.extend(entries)

    # Convert list of dictionaries to DataFrame
    df = pd.DataFrame(all_dicts)

    # Write the DataFrame to a CSV file
    df.to_csv(output_filename, index=False)
    
    

def json_to_tree_data(data: dict):
    def item_to_list_of_entries(item) -> list[dict]:
        surrounding_information = {prop: item[prop]
                                   for prop in ["problem_name", "pRef_method"]}
        surrounding_information = {"problem": item["problem_name"],
                                   "pRef_method": item["pRef_method"]}

        entries = item["results_by_tree"]
        entries = [thing for thing in entries if "order_tree" in thing]  

        def convert_order_tree(order_tree, accumulator = None, current_depth: int = 0):
            if accumulator is None:
                accumulator = defaultdict(list)
            accumulator[current_depth].append(order_tree["own"])
            if len(order_tree["matching"]) > 0:
                convert_order_tree(order_tree["matching"], accumulator, current_depth+1)

            if len(order_tree["unmatching"]) > 0:
                convert_order_tree(order_tree["unmatching"], accumulator, current_depth+1)

            return accumulator
        def convert_tree_to_averages_by_level(entry):
            ps_search_info = {prop: entry[prop]
                                   for prop in ["ps_budget", "ps_population", "metrics"]}
            tree_structure = entry["order_tree"]
            just_depths = convert_order_tree(tree_structure)
            average_orders_by_depth = {f"average_at_{depth}": np.average(orders)
                              for depth, orders in just_depths.items()}
            standard_deviations = {f"sd_at_{depth}": np.std(orders)
                              for depth, orders in just_depths.items()}
            overall_average = {"overall_average": np.average(list(itertools.chain(*(just_depths.values()))))}
            return surrounding_information | ps_search_info | average_orders_by_depth | overall_average | standard_deviations


        entries = list(map(convert_tree_to_averages_by_level, entries))
        return entries

    return [entry for item in data for entry in item_to_list_of_entries(item)]

def convert_tree_data_to_df(input_directory, output_filename):

    all_dicts = []
    # Iterate through all files in the input directory
    for filename in os.listdir(input_directory):
        # Construct full file path
        file_path = os.path.join(input_directory, filename)

        # Check if the file is a JSON file
        if not os.path.isfile(file_path):
            continue

        if not is_valid_data_file(file_path):
            continue


        with open(file_path, 'r') as file:
            data = json.load(file)
            entries = json_to_tree_data(data)
            all_dicts.extend(entries)

    # Convert list of dictionaries to DataFrame
    df = pd.DataFrame(all_dicts)

    # Write the DataFrame to a CSV file
    df.to_csv(output_filename, index=False)
    
    

def prettify_kind_column(df):
    df['kind'] = df.apply(
    lambda row: (
        'PS-W' if row['kind'] == 'ps' and row['metrics'] == 'variance' else
        'PS-WA' if row['kind'] == 'ps' else
        'Trad.' if row['kind'] == 'naive' else
        'IAI' if row['kind'] == 'iai' else
        row['kind']
    ),
    axis=1
)
    
    

    
    

In [2]:
#run_location = r"/Users/gian/Desktop/CondorResults/VDT/compareown/run3/"
run_location = r"C:\Users\gac8\Desktop\CondorResults\VDT\compareown\all_final_runs"

results_csv = os.path.join(run_location, "results.csv")
tree_data_csv = os.path.join(run_location, "tree_data.csv")


convert_accuracy_data_to_df(os.path.join(run_location, "data"), results_csv)
convert_tree_data_to_df(os.path.join(run_location, "data"), tree_data_csv)


In [56]:

accuracy_data = pd.read_csv(results_csv)
tree_data = pd.read_csv(tree_data_csv)

accuracy_data["depth"] = accuracy_data["depth"].apply(str)
display(accuracy_data)
display(accuracy_data.dtypes)
display(tree_data)

prettify_kind_column(accuracy_data)
#prettify_kind_column(tree_data)


for kind in accuracy_data["kind"].unique():
    matching_rows = accuracy_data[accuracy_data['kind'] == kind]
    print(f"For the tree kind {kind}, there are {matching_rows.shape[0]} rows")

#headers = "kind,depth,ps_budget,ps_population,avoid_ancestors,metrics,problem,pRef_method,mse,mae,r_sq,evs"

,kind,depth,problem,pRef_method,pRef_size,mse,mae,r_sq,evs,ps_budget,ps_population,avoid_ancestors,metrics,cp
0,naive,2,SAT_S,uniform,10000,10.428119,2.562856,0.244222,0.244534,NaN,NaN,NaN,NaN,NaN
1,naive,3,SAT_S,uniform,10000,8.912910,2.392481,0.354036,0.354210,NaN,NaN,NaN,NaN,NaN
2,naive,4,SAT_S,uniform,10000,7.868440,2.238714,0.429734,0.430817,NaN,NaN,NaN,NaN,NaN
3,naive,5,SAT_S,uniform,10000,6.944226,2.096012,0.496717,0.497513,NaN,NaN,NaN,NaN,NaN
4,naive,6,SAT_S,uniform,10000,6.355751,2.011702,0.539367,0.539568,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
87235,iai,2,BT,Tabu,10000,30.296781,4.379711,0.136241,0.136257,NaN,NaN,NaN,NaN,0.25
87236,iai,3,BT,Tabu,10000,27.494236,4.131999,0.216141,0.216218,NaN,NaN,NaN,NaN,0.25
87237,iai,4,BT,Tabu,10000,24.733129,3.867284,0.294860,0.294862,NaN,NaN,NaN,NaN,0.25
87238,iai,5,BT,Tabu,10000,21.599822,3.706826,0.384191,0.384239,NaN,NaN,NaN,NaN,0.25


kind                object
depth               object
problem             object
pRef_method         object
pRef_size            int64
mse                float64
mae                float64
r_sq               float64
evs                float64
ps_budget          float64
ps_population      float64
avoid_ancestors     object
metrics             object
cp                 float64
dtype: object

,problem,pRef_method,ps_budget,ps_population,metrics,average_at_0,average_at_1,average_at_2,average_at_3,average_at_4,average_at_5,overall_average,sd_at_0,sd_at_1,sd_at_2,sd_at_3,sd_at_4,sd_at_5
0,SAT_S,uniform,5000,100,variance,1.0,1.5,2.25,3.250,4.125000,5.103448,4.216667,0.0,0.5,0.829156,1.561249,1.690969,2.202582
1,SAT_S,uniform,5000,100,variance estimated_atomicity,1.0,1.5,1.75,1.750,1.875000,1.937500,1.857143,0.0,0.5,0.433013,0.433013,0.330719,0.242061
2,SAT_S,uniform,5000,100,variance,2.0,1.5,2.25,3.750,3.562500,3.516129,3.387097,0.0,0.5,0.433013,1.639360,1.498697,1.965361
3,SAT_S,uniform,5000,100,variance estimated_atomicity,2.0,1.0,1.50,2.000,1.812500,1.937500,1.857143,0.0,0.0,0.500000,0.000000,0.390312,0.242061
4,SAT_S,GA,5000,100,variance,4.0,4.5,5.00,5.875,5.357143,5.840000,5.574074,0.0,2.5,1.224745,2.570870,2.438007,2.781079
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10747,BT,SA,5000,100,variance estimated_atomicity,2.0,2.0,2.00,1.875,2.875000,2.206897,2.316667,0.0,0.0,0.000000,0.599479,3.314268,1.423432
10748,BT,Tabu,5000,100,variance,8.0,14.0,12.50,12.250,9.133333,8.000000,9.528302,0.0,5.0,0.866025,6.437197,4.814792,6.460583
10749,BT,Tabu,5000,100,variance estimated_atomicity,2.0,2.0,2.00,1.875,1.928571,2.681818,2.254902,0.0,0.0,0.000000,0.330719,0.593330,1.394351
10750,BT,Tabu,5000,100,variance,13.0,5.5,7.00,15.500,11.375000,9.320000,10.553571,0.0,4.5,3.605551,7.245688,6.489174,4.913003


For the tree kind Trad., there are 26880 rows
For the tree kind PS-W, there are 26880 rows
For the tree kind PS-WA, there are 26880 rows
For the tree kind IAI, there are 6600 rows


In [57]:
depth = "6"
pRef_size = 10000
independent_variables = ["pRef_method", "kind"]
dependent_variables = ["r_sq"]

usable_data = accuracy_data.copy()

usable_data = usable_data[usable_data["depth"] == depth]
usable_data = usable_data[usable_data["pRef_size"] == pRef_size] 

for problem in accuracy_data["problem"].unique():
    print(f"For the problem {problem}")
    rows_with_right_problem = usable_data[usable_data["problem"] == problem]
    tabulated = get_mean_for_combinations(rows_with_right_problem, independent_variables, dependent_variables)
    #tabulated.sort_values(by=dependent_variables, ascending=True, inplace=True)
    
    pivot_table = tabulated.pivot_table(index = independent_variables[0], columns = independent_variables[1:], values = dependent_variables[0])
    #pivot_table.sort_index(axis=0).sort_index(axis=1)
    pivot_table_custom_order = pivot_table.reindex(columns=['PS-W', 'PS-WA', 'IAI', 'Trad.'])


    def bold_max(s):
        return ['font-weight: bold' if v == s.max() else '' for v in s]

    # Apply the style to the pivot table
    styled_table = pivot_table.style.apply(bold_max, axis=1)

    #display(tabulated)
    display(pivot_table)



For the problem SAT_S


kind,IAI,PS-W,PS-WA,Trad.
pRef_method,,,,
GA,0.525274,0.618580,0.485401,0.562372
SA,0.539704,0.593804,0.501453,0.601810
Tabu,0.337426,0.423860,0.312834,0.375588
uniform,0.478857,0.497186,0.456755,0.527536


For the problem SAT_M


kind,IAI,PS-W,PS-WA,Trad.
pRef_method,,,,
GA,0.660700,0.828681,0.673486,0.698873
SA,0.630146,0.687635,0.573276,0.718355
Tabu,0.253814,0.336812,0.191867,0.313599
uniform,0.213014,0.202593,0.129084,0.236551


For the problem SAT_L


...

kind,IAI,PS-W,PS-WA,Trad.
pRef_method,,,,
GA,0.844046,0.908386,0.853384,0.864045
SA,0.882817,0.813227,0.780825,0.907927
Tabu,0.532105,0.373592,0.391133,0.617719
uniform,-0.001568,-0.006810,-0.017420,-0.055156


For the problem GC_L


kind,IAI,PS-W,PS-WA,Trad.
pRef_method,,,,
GA,0.778102,0.855869,0.795863,0.797102
SA,0.728527,0.642769,0.609790,0.793958
Tabu,0.306978,0.184448,0.218185,0.393329
uniform,0.003746,-0.006997,-0.016037,-0.032888


For the problem BT


kind,IAI,PS-W,PS-WA,Trad.
pRef_method,,,,
GA,0.873823,0.930585,0.855490,0.887955
SA,0.937585,0.922383,0.859921,0.953010
Tabu,0.523403,0.535108,0.365035,0.579886
uniform,0.183650,0.131252,0.063303,0.188191


In [122]:
pRef_size = 10000
independent_variables = ["problem", "pRef_method", "kind", "depth"]
dependent_variables = ["r_sq"]

usable_data = accuracy_data.copy()
usable_data = usable_data[usable_data["pRef_size"] == pRef_size] 


def style_pivot_table(pivot_table):
    custom_column_order = ['PS-W', 'PS-WA', 'IAI', 'Trad.']

    # Reorder columns based on custom order
    pivot_table = pivot_table.reindex(columns=custom_column_order)

    pivot_table = pivot_table.mul(100).round(2).astype(str) + "%"

    styled_df = pivot_table.style.apply(bold_max, axis=1)
    return styled_df

def fix_latex(input_string):
    # Replace '%' with '\%'
    replacements = {"%":"\\%",
                    "pRef_method":"Met.",
                    "SAT_": "SAT\_",
                    "GC_L": "GC\_anna",
                    "GC_S": "GC\_jean",
                   "\\font-weightbold": ""}
    modified_string = str(input_string)
    for orig, replacement in replacements.items():
        modified_string = modified_string.replace(orig, replacement)
    
    return modified_string

def bold_max(s):
    return ['font-weight: bold' if v == s.max() else '' for v in s]

for problem in usable_data['problem'].unique():
    with_right_problem = usable_data[usable_data['problem'] == problem]
    pivot_table = tabulated.pivot_table(index = ["pRef_method", "depth"], 
                                        columns = ["kind"], 
                                        values = dependent_variables[0])

    pivot_table = style_pivot_table(pivot_table)
    print(fix_latex(f"{problem = }"))
    latex_text = pivot_table.to_latex()
    latex_text = fix_latex(latex_text)
    print(latex_text)
    
    # Display the styled dataframe
    # display(styled_df)
    
    #display(pivot_table)

problem = 'SAT\_S'
\begin{tabular}{llllll}
 & kind & PS-W & PS-WA & IAI & Trad. \\
Metaheuristic & depth &  &  &  &  \\
\multirow[c]{5}{*}{GA} & 2 &  70.04\% & 47.69\% & 33.7\% & 51.48\% \\
 & 3 &  76.88\% & 58.46\% & 51.71\% & 62.1\% \\
 & 4 &  80.39\% & 65.64\% & 62.68\% & 68.98\% \\
 & 5 &  82.65\% & 70.64\% & 69.86\% & 73.71\% \\
 & 6 &  84.05\% & 74.02\% & 74.07\% & 76.76\% \\
\multirow[c]{5}{*}{SA} & 2 &  51.5\% & 28.8\% & 20.74\% & 38.48\% \\
 & 3 &  59.96\% & 40.59\% & 40.03\% & 53.13\% \\
 & 4 &  66.05\% & 51.26\% & 56.27\% & 64.8\% \\
 & 5 & 70.64\% & 59.99\% & 67.87\% &  74.01\% \\
 & 6 & 74.25\% & 67.46\% & 75.55\% &  80.8\% \\
\multirow[c]{5}{*}{Tabu} & 2 & 15.85\% & 6.48\% &  7.03\% & 12.58\% \\
 & 3 &  21.88\% & 10.62\% & 13.6\% & 19.19\% \\
 & 4 &  27.33\% & 15.82\% & 21.33\% & 27.05\% \\
 & 5 & 32.38\% & 21.88\% & 29.49\% &  36.0\% \\
 & 6 & 37.14\% & 28.87\% & 38.3\% &  45.43\% \\
\multirow[c]{5}{*}{uniform} & 2 & 7.67\% & 5.07\% & 4.14\% &  8.29\% \\
 & 3 & 10.34\% &

In [5]:

ps_indep_vars = ["problem", "pRef_method", "metrics"]

tabulated_accuracy_results = get_mean_for_combinations(accuracy_data, 
                                                       ps_indep_vars, 
                                                       dependent_variables)
tabulated_tree_results = get_mean_for_combinations(tree_data, 
                                                   ps_indep_vars, 
                                                   ["overall_average"])

matched = tabulated_accuracy_results.merge(tabulated_tree_results, on=ps_indep_vars)
# matched = matched[matched["pRef_method"] == "GA"]

for problem in sorted(matched["problem"].unique()):
    print(f"For problem {problem}")
    for_problem = matched[matched['problem'] == problem]
    display(for_problem)
    
    
#display(matched)


# for problem in matched["problem"].unique():
#     with_correct_problem = matched[matched["problem"]==problem]
#     print(f"for the problem {problem}")
#     utils.simple_scatterplot(x_label="accuracy (R^2)",
#                              y_label="complexity",
#                              xs = with_correct_problem["r_sq"],
#                              ys = with_correct_problem["overall_average"])







For problem BT


,problem,pRef_method,metrics,r_sq,overall_average
0,BT,GA,variance,0.908832,24.949370
1,BT,GA,variance estimated_atomicity,0.770694,2.027426
2,BT,SA,variance,0.802369,8.388082
3,BT,SA,variance estimated_atomicity,0.617541,2.674918
4,BT,Tabu,variance,0.354280,10.262879
5,BT,Tabu,variance estimated_atomicity,0.180751,2.314221
6,BT,uniform,variance,0.121704,7.575813
7,BT,uniform,variance estimated_atomicity,0.052709,1.923834


For problem GC_L


,problem,pRef_method,metrics,r_sq,overall_average
8,GC_L,GA,variance,0.847352,17.312544
9,GC_L,GA,variance estimated_atomicity,0.721254,1.983524
10,GC_L,SA,variance,0.414686,6.531048
11,GC_L,SA,variance estimated_atomicity,0.344832,2.404713
12,GC_L,Tabu,variance,0.087855,7.506091
13,GC_L,Tabu,variance estimated_atomicity,0.082965,2.265616
14,GC_L,uniform,variance,-0.002415,5.900513
15,GC_L,uniform,variance estimated_atomicity,-0.004254,2.016559


For problem GC_S


...

,problem,pRef_method,metrics,r_sq,overall_average
24,SAT_L,GA,variance,0.871158,23.935699
25,SAT_L,GA,variance estimated_atomicity,0.677783,2.103545
26,SAT_L,SA,variance,0.606348,8.351538
27,SAT_L,SA,variance estimated_atomicity,0.389902,2.299923
28,SAT_L,Tabu,variance,0.226575,9.712769
29,SAT_L,Tabu,variance estimated_atomicity,0.098289,2.123897
30,SAT_L,uniform,variance,0.081032,5.699044
31,SAT_L,uniform,variance estimated_atomicity,0.044300,1.971372


For problem SAT_M


,problem,pRef_method,metrics,r_sq,overall_average
32,SAT_M,GA,variance,0.782358,13.807841
33,SAT_M,GA,variance estimated_atomicity,0.555318,2.001298
34,SAT_M,SA,variance,0.523574,6.922146
35,SAT_M,SA,variance estimated_atomicity,0.313858,2.051938
36,SAT_M,Tabu,variance,0.228453,8.427194
37,SAT_M,Tabu,variance estimated_atomicity,0.095374,2.037098
38,SAT_M,uniform,variance,0.168090,5.368040
39,SAT_M,uniform,variance estimated_atomicity,0.102274,1.924268


For problem SAT_S


,problem,pRef_method,metrics,r_sq,overall_average
40,SAT_S,GA,variance,0.518474,5.453692
41,SAT_S,GA,variance estimated_atomicity,0.363288,1.869814
42,SAT_S,SA,variance,0.465693,4.620950
43,SAT_S,SA,variance estimated_atomicity,0.349195,1.874325
44,SAT_S,Tabu,variance,0.313115,5.102003
45,SAT_S,Tabu,variance estimated_atomicity,0.206611,1.904963
46,SAT_S,uniform,variance,0.388592,3.763063
47,SAT_S,uniform,variance estimated_atomicity,0.343887,1.799370
